# Acoustic finite-difference wavefield lab

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mosesfalowo/ogbonlab-notebooks/blob/main/notebooks/ssm_acoustic_fd_lab.ipynb)


Generated by [OgbonLab](https://ogbonlab.com), Synthetic Seismic Modeling course.

**Assumptions**: Course defaults. Edit the parameters cell and run again; every value is yours to change.

**Workflow**:
1. 10.2 Subtle Stratigraphic Trap
2. 5.1 Diffractions
3. 5.6 Convolution vs the Wave Equation
4. 9.6 Model QC

In [ ]:
# Ask Ogbon starter: Image a subtle or complex structure
# Generated by OgbonLab (ogbonlab.com), Synthetic Seismic Modeling course.
"""What a scatterer or a reflector adds to a shot record, by the 2D acoustic wave equation.

Follows Sections 4.2 to 4.6 (stencils, the CFL limit, dispersion, absorbing
boundaries, shot records) and 5.1 (Diffractions); the cell is Alford, Kelly and Boore (1974).
Units: m, m/s, ms (the solver itself steps in s), Hz.
Assumptions and limits:
- No density (Section 6.6): reflections come from the velocity contrast alone,
  so read timing and shape, not amplitude; acoustic, so no shear waves.
- 2D: a point source here is a line source in 3D, so amplitudes fall as one
  over the square root of distance and the pulse keeps a 2D tail.
- Fourth order in space, second in time, stable to Courant 0.612; np.roll wraps
  the edges and the 40-cell sponge damps, not removes, what reaches them.
- The full job is counted, not run. What runs is one shot through a window of the
  field earth, its cell set by the same rule at a peak of at most FIELD_F_MAX_HZ,
  so that it runs in seconds.

It prints the numbers Figure 12.5 computes for the same answers as name = value
lines; then it builds the field earth every starter shares, prints what it measures
there as [field] name = value lines, and draws one figure. Python 3.9+, numpy and
matplotlib; the earth's random numbers are seeded, so every run prints the same.
Edit the parameters and run again.
"""
import numpy as np
import matplotlib.pyplot as plt

# --- parameters from your interview ---
TARGET = 'pinchout'  # 'pinchout', 'fault' or 'layers'
DIFFRACTION = 'no'   # 'yes' when the diffraction itself is the data
F_PEAK_HZ = 30.0     # Ricker peak frequency (Hz)
GRID_N = 300         # grid points per side of the full job
N_SHOTS = 1          # shots in the full job
BUDGET = 'laptop'    # the compute you said you have

In [ ]:
# --- the book's constants (not interview answers) ---
# the book's standard rocks (SSM.STANDARD, Section 3.3): P velocity (m/s), S velocity (m/s), density (g/cc)
ROCKS = {
    'shale': (2800, 1241.3793103448277, 2.2521031547808517),
    'brine sand': (2889.973997793982, 1504.4719169684909, 2.2545487499999997),
    'oil sand': (2543.1993596875604, 1529.5017037311773, 2.1813626422963988),
    'gas sand': (2341.92404704112, 1582.4208998550723, 2.0379042775100316)
}
VP_MIN = min(r[0] for r in ROCKS.values())      # the slowest P wave sets the cell (m/s)
VP_MAX = max(r[0] for r in ROCKS.values())      # the fastest sets the time step (m/s)
COURANT = np.sqrt(3.0 / 8.0)    # the fourth-order stability limit in 2D, 0.612 (Section 4.3)
NPTS = 9                        # points the fourth-order Laplacian reads in 2D
PPW = 5.0                       # points per wavelength at the upper half-power frequency (Section 4.4)
N_ABS, SPONGE_STRENGTH = 40, 0.008   # Cerjan's sponge: width (cells) and strength
FIELD_F_MAX_HZ, FIELD_T_MS, FIELD_DEPTH_M = 25.0, 1000.0, 1100.0   # the field shot: peak at most (Hz), record (ms), depth (m)
WINDOWS = {'pinchout': (0.22, 0.62), 'fault': (0.5, 0.9), 'layers': (0.0, 0.4), 'shots': (0.15, 0.85)}   # x / length
SMOOTH_STEPS = 80               # the smoothing of the earth whose record is subtracted
MUTE_MPS, MUTE_MS = 1600.0, 80.0   # the difference is shown below the direct wave: offset / speed + a margin
SNAP_MS = 380.0                 # the wavefield's snapshot, as the wave crosses the crest (ms)
WINDOW, TITLE_BG = TARGET, 'the smoothed earth'
CLIP = 0.8                      # display clip, a share of the largest amplitude

def report(name, value, fmt='%.6f'):
    """Print one result as name = value, the line Figure 12.5 checks against the book's kernel."""
    print(('%s = ' + fmt) % (name, value))

def style(ax, title, xlabel, ylabel, grid=True):
    """The starters' one look: a title, both axes labelled with their units, a light grid."""
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if grid:
        ax.grid(True, alpha=0.3)

# --- the Ogbon field earth: one seeded 2D earth that every starter shares ---
# In geological order: thin sand and shale beds with a regional dip, folded into an
# asymmetric anticline and cut by normal faults, the largest a listric growth fault
# (its throw grows with depth and its hanging wall rolls over into it); uplift, and an
# angular unconformity that truncates the crest; the reservoir sand subcrops beneath
# it, wedging out against it, shaling out to the left, sealed by a shale cap and
# holding gas above a flat contact; then a draped overburden with a channel. Each
# cell's velocity follows the
# compaction trend at its deepest burial (the rocks under the unconformity were once
# buried deeper), sands soft when shallow and hard when deep; Gardner's density; the
# mudrock and sandstone lines for S. Units: m, m/s, g/cc; times in ms.
FAULTS = [(0.66, 1.0, 1.4, 380.0), (0.80, -1.0, 0.6, 470.0), (0.22, -1.0, 0.5, 560.0)]   # (x / length, side, size, tip m)
COMPACTION = (1900.0, 1.4)                             # P velocity = 1900 m/s + 1.4 per metre of burial, before lithology
K_RES = 9                                              # the reservoir: bed 9 of the deep package, 70 m thick
SEAL_M = 30.0                                          # the shale cap on the reservoir's top: 30 m, and never under two cells

def field_earth(nx=240, nz=320, dx=15.0, dz=5.0, seed=7, crest=0.45, fold=1.0, dip=1.0, faults=None, throw=1.0,
                unconformity=True, seal=None, upper=None, lower=None, column_m=130.0):
    """The field earth on nz by nx cells of dz by dx (m). seal, upper and lower are (vp, vs, density) of the shale on the
    unconformity and of the reservoir above and below its flat contact, or None for the compaction trend's own rocks."""
    rng = np.random.default_rng(seed)
    x, z = np.arange(nx) * dx, np.arange(nz) * dz
    xn, zz, sc = x / x[-1], z[:, None] + 0.0 * x, nz * dz / 1600.0     # sc: the geometry scales with the grid's depth
    hump = np.exp(-0.5 * ((xn - crest) / np.where(xn < crest, 0.13, 0.25)) ** 2)   # the anticline, steep to the left
    swell = 1.0 + 0.15 * np.sin(2 * np.pi * (0.8 * xn + 0.37 * np.arange(26)[:, None]))   # each bed thickens and thins
    thick = sc * (24.0 + 22.0 * rng.random(26))[:, None] * swell
    thick[K_RES] = sc * 70.0
    beds = sc * 470.0 + np.cumsum(thick, axis=0) - thick + sc * (170.0 * dip * xn - 300.0 * fold * hump)
    if faults is not None:
        fs = [(rng.uniform(0.12, 0.88), rng.choice([-1.0, 1.0]), rng.uniform(0.4, 1.4), rng.uniform(300.0, 600.0))
              for k in range(faults)]
    fault = np.zeros((nz, nx), bool)
    for x0, side, size, tip in (FAULTS if faults is None else fs):
        tip, a = tip * sc, 0.2 / (1200.0 * sc) ** 1.3       # the plane: x / length = x0 + side a (depth - tip) to the 1.3, listric
        plane = np.where(side * (xn - x0) > 0, tip + (np.abs(xn - x0) / a) ** (1 / 1.3), -np.inf)   # its depth at each x
        for k in range(26):
            g = throw * size * sc * 90.0 * np.clip((beds[k] - tip) / (700.0 * sc), 0.0, 1.6)      # growth: deeper beds slip more
            xk = x0 + side * a * np.maximum(beds[k] - tip, 0.0) ** 1.3                          # where the plane cuts the bed
            g = g * (1.0 + 0.6 * np.exp(-np.abs(xn - xk) / 0.04))                                # the rollover into the fault
            beds[k] = beds[k] + np.clip(plane - beds[k], 0.0, g)       # the hanging wall drops; in the heave the bed lies on the plane
        ix = np.rint((x0 + side * a * np.maximum(z - tip, 0.0) ** 1.3) * (nx - 1)).astype(int)
        on = (z > tip) & (ix >= 0) & (ix < nx)
        fault[np.nonzero(on)[0], ix[on]] = True
    beds = np.maximum.accumulate(beds, axis=0)
    if unconformity:                                   # cut just under the reservoir's crest, so it subcrops on both flanks
        unc = beds[K_RES + 1, np.argmax(hump)] + sc * (15.0 + 160.0 * dip * (xn - crest)
                                                       + 10.0 * np.sin(2 * np.pi * (1.3 * xn + 0.2)))
    else:
        unc = beds[0] - sc * 30.0                      # no erosion: the overburden lies on the beds
    fault &= zz > unc                                  # the faults are older than the unconformity
    fr = np.cumsum(0.6 + rng.random(13))
    fr = (fr - fr[0]) / (fr[-1] - fr[0])
    over = [sc * 40.0 + (unc.min() - sc * 90.0) * f + 0.85 * f * f * (unc - unc.min()) - sc * 25.0 * f * hump for f in fr]
    tops = np.maximum.accumulate(np.vstack(over + list(np.maximum(beds, unc))), axis=0)
    lay = (zz[:, None, :] >= tops[None, :, :]).sum(axis=1)                    # layer 13 is the seal; bed k is layer 14 + k
    sand = np.where(lay < 13, rng.random(41)[lay] < 0.45, (lay - 14) % 2 == 1) & (lay > 0) & (lay != 13)
    chan = (zz >= over[5]) & (zz < over[5] + sc * 45.0 * np.exp(-0.5 * ((xn - 0.7) / 0.04) ** 2))
    res = lay == 14 + K_RES
    top = np.where(res, zz, np.inf).min(axis=0)
    cap = (zz < top) & (zz >= top - max(SEAL_M * sc, 2.0 * dz))   # the seal: shale on the sand's top, under a bed or the unconformity
    sand = np.where(cap, False, sand)
    burial = zz + np.where(lay >= 14, sc * 150.0 * hump, 0.0)                # uplift: the deep rocks were buried deeper
    vary = 1.0 + 0.035 * np.clip(rng.standard_normal(41), -1.2, 1.2)        # bed to bed variety
    vp = (COMPACTION[0] + COMPACTION[1] * burial) * vary[lay]
    vp = np.where(sand | chan, vp * (0.86 + 0.1 * np.minimum(burial / 1000.0, 1.8)), vp)
    vp = np.where(chan, vp * 0.92, vp)                 # the channel cuts a horizon and is filled with a soft sand
    sand |= chan
    vs = np.where(sand, 0.804 * vp - 856.0, (vp - 1360.0) / 1.16)            # the sandstone and mudrock lines (Castagna)
    den = 1.741 * (vp / 1000.0) ** 0.25                                      # Gardner
    ntg = np.clip((xn - 0.12) / 0.12, 0.0, 1.0)                              # the reservoir shales out to the left
    contact = float(top.min()) + column_m * sc         # the flat contact, column_m under the shallowest sand
    for rock, on in ((seal, (lay == 13) | cap), (upper, res & (zz < contact)), (lower, res & (zz >= contact))):
        if rock is not None:
            w = 1.0 if rock is seal else ntg
            vp, vs, den = [np.where(on, w * rock[i] + (1 - w) * q, q) for i, q in enumerate((vp, vs, den))]
    return dict(x=x, z=z, vp=vp, vs=vs, den=den, lay=lay, sand=sand, res=res, fault=fault, unc=unc, top=top,
                contact=contact, ntg=ntg, dx=dx, dz=dz, chan=chan, cap=cap, gas=res & (zz < contact) & (ntg > 0.5))

def field_time(m, dt_ms, t_max_ms):
    """Each trace from depth to two-way time by vertical rays: every cell array of m on a dt_ms grid to t_max_ms."""
    t_top = 2000.0 * np.cumsum(m['dz'] / m['vp'], axis=0) - 2000.0 * m['dz'] / m['vp']
    t = np.arange(0.0, t_max_ms, dt_ms)
    pick = np.array([np.searchsorted(t_top[:, i], t, 'right') - 1 for i in range(t_top.shape[1])]).T
    cols = np.arange(t_top.shape[1])[None, :]
    out = dict((k, m[k][pick, cols]) for k in ('vp', 'vs', 'den', 'lay', 'sand', 'res', 'gas'))
    seen = np.vstack([0 * cols, np.cumsum(m['fault'], axis=0)])                 # fault cells above each row
    after = np.maximum(np.vstack([pick[1:], pick[-1:] + 1]), pick + 1)
    out['fault'] = seen[np.minimum(after, seen.shape[0] - 1), cols] > seen[pick, cols]   # a fault cell inside the sample
    out['t'], out['t_top'] = t, t_top
    return out

def reflectivity(ai):
    """Normal-incidence coefficient at the top of every sample, down axis 0 (Section 2.1)."""
    rc = np.zeros_like(ai)
    rc[1:] = (ai[1:] - ai[:-1]) / (ai[1:] + ai[:-1])
    return rc

def q_peak_hz(f0, t_s, q):
    """Peak (Hz) of a Ricker of peak f0 after t_s seconds through constant Q: the root of 2 f f / (f0 f0) + (pi t / Q) f = 2."""
    b = np.pi * t_s / q
    return (-b + np.sqrt(b * b + 16.0 / f0 ** 2)) * f0 ** 2 / 4.0

def field_image(rc, dt_ms, f0, q=80.0, snr=None, seed=11):
    """Every trace convolved with a zero-phase Ricker whose peak falls with time as constant q says, plus coloured noise."""
    nt, dt = rc.shape[0], dt_ms * 1e-3
    f = np.fft.rfftfreq(2 * nt, dt)[:, None]
    def band(fp):                                      # the Ricker's spectrum (Section 1.1), over dt for a sum of samples
        return 2.0 / np.sqrt(np.pi) * f ** 2 / fp ** 3 * np.exp(-(f / fp) ** 2) / dt
    def filt(a, fp):
        return np.fft.irfft(np.fft.rfft(a, 2 * nt, axis=0) * band(fp), 2 * nt, axis=0)[:nt]
    fp = q_peak_hz(f0, np.arange(nt) * dt, q)
    knots = np.linspace(fp.min(), fp.max(), 4 if fp.max() - fp.min() > 1e-9 else 1)
    step = max((fp.max() - fp.min()) / 3.0, 1e-9)
    out = sum(np.clip(1.0 - np.abs(fp - k) / step, 0.0, 1.0)[:, None] * filt(rc, k) for k in knots)
    if snr:
        noise = filt(np.random.default_rng(seed).standard_normal(rc.shape), fp.mean())
        noise = (noise + np.roll(noise, 1, 1) + np.roll(noise, -1, 1)) / 3.0          # coherent over a few traces
        out = out + noise * np.sqrt((out ** 2).mean() / (noise ** 2).mean()) / snr
    return out

def fieldnote(name, value, fmt='%.3f'):
    """Print a number of the field earth as [field] name = value: computed here, not by Figure 12.5."""
    print(('[field] %s = ' + fmt) % (name, value))

def show(ax, a, x_m, y, title, ylabel='two-way time (ms)', clip=None):
    """A section as an image, black for positive, clipped at clip (or at its 99th percentile): km across, time or depth down."""
    c = clip if clip else np.percentile(np.abs(a), 99.0) + 1e-12
    ax.imshow(a, cmap='gray_r', vmin=-c, vmax=c, aspect='auto', extent=[x_m[0] / 1e3, x_m[-1] / 1e3, y[-1], y[0]])
    style(ax, title, 'distance (km)', ylabel, grid=False)

def upper_half_power_ratio():
    """The Ricker's upper half-power frequency over its peak: u exp(1 - u) = 1 / root 2, u = (f / fp) squared."""
    u = 2.0
    for _ in range(60):                          # Newton's method
        F = u * np.exp(1 - u) - 1 / np.sqrt(2)
        nu = u - F / ((1 - u) * np.exp(1 - u))
        if abs(nu - u) < 1e-15:
            u = nu
            break
        u = nu
    return np.sqrt(u)

def alford_h(v_min, f_peak):
    """Cell (m): PPW points per wavelength at the upper half-power frequency (Alford, Kelly and Boore 1974)."""
    return v_min / (PPW * f_peak * upper_half_power_ratio())

def job_size(n, f_peak, v_min, v_max, shots):
    """Cell (m), step (ms), steps, cells and multiply-adds of n-by-n runs, counted as Figure 0.2 counts."""
    h = alford_h(v_min, f_peak)
    t_ms = 2000.0 * n * h / v_min            # two-way time to the bottom at the slowest P speed
    dt_ms = 1000.0 * COURANT * h / v_max     # at the stability limit (Section 4.3)
    steps = int(np.ceil(t_ms / dt_ms - 1e-9))
    cells = n * n
    return h, dt_ms, steps, cells, cells * steps * NPTS * shots

def d2(u, axis):
    """Fourth-order second difference in units of the cell (Section 4.2); np.roll wraps the edges."""
    return (-(np.roll(u, 2, axis) + np.roll(u, -2, axis)) / 12.0
            + 4.0 / 3.0 * (np.roll(u, 1, axis) + np.roll(u, -1, axis)) - 2.5 * u)

def sponge(nz, nx, n_abs=N_ABS, strength=SPONGE_STRENGTH):
    """Cerjan's absorbing border (Section 4.5): 1 inside, tapering towards the edges."""
    def profile(n):
        d = np.minimum(np.arange(n), np.arange(n)[::-1])
        return np.where(d < n_abs, np.exp(-(strength * (n_abs - d)) ** 2), 1.0)
    return np.outer(profile(nz), profile(nx)) if nx > 2 * n_abs else profile(nz)[:, None]

def source_derivative(f_peak, dt, nt):
    """Time derivative of a Ricker delayed one period (dt in s): a plane wave then has the Ricker's shape."""
    n0 = int(round((1.0 / f_peak) / dt))
    b = np.pi ** 2 * f_peak ** 2
    tt = (np.arange(nt) - n0) * dt
    return 2 * b * tt * (2 * b * tt * tt - 3) * np.exp(-b * tt * tt), n0

def run(v, h, dt, nt, src_iz, src_ix, src, rec_iz, rec_ix, frames=None, every=0):
    """Leapfrog the constant-density acoustic wave equation; v in m/s, h in m, dt in s; returns the record.
    Given a list frames, it also keeps a copy of the wavefield every every steps."""
    courant = v.max() * dt / h
    if courant > COURANT * (1 + 1e-9):
        raise ValueError('Courant number %.3f is past the 2D fourth-order limit %.3f (Section 4.3): shrink dt' % (courant, COURANT))
    nz, nx = v.shape
    c2 = (v * dt / h) ** 2
    damp = sponge(nz, nx)
    u0 = np.zeros((nz, nx)); u1 = np.zeros((nz, nx))
    rec = np.zeros((nt, np.size(rec_ix) if np.ndim(rec_ix) else 1))
    for it in range(nt):
        u2 = 2 * u1 - u0 + c2 * (d2(u1, 0) + d2(u1, 1))
        u2[src_iz, src_ix] += src[it] * c2[src_iz, src_ix]
        u2 *= damp; u1 *= damp
        rec[it] = u2[rec_iz, rec_ix]
        if frames is not None and it % every == 0:
            frames.append(u2.copy())
        u0, u1 = u1, u2
    return rec

def plane_wave_check(v1, v2, f_peak, h, dt_ms):
    """The same stencil on one column: a plane wave meets an interface 70 cells below the source; ms."""
    nz, iz_s, iz_i = 200, 50, 120
    dt = dt_ms * 1e-3 / 4.0          # a quarter of the job's step: the leapfrog's time error shrinks 16 times
    pred = (iz_i - iz_s) * h / v1 * 2
    nt = int((pred + 3.0 / f_peak) / dt) + 1
    src, n0 = source_derivative(f_peak, dt, nt)
    v = np.full((nz, 1), float(v1)); w = v.copy(); w[iz_i:] = v2
    r_h = run(v, h, dt, nt, iz_s, 0, src, iz_s, 0)
    r_w = run(w, h, dt, nt, iz_s, 0, src, iz_s, 0)
    refl = (r_w - r_h)[:, 0]
    return 1000.0 * int(np.abs(refl).argmax()) * dt, 1000.0 * (n0 * dt + pred)

def validate(grid_n, f_peak, n_shots):
    """Stop with a clear message on a parameter the model cannot use."""
    if grid_n < 1 or n_shots < 1 or f_peak <= 0:
        raise ValueError('GRID_N and N_SHOTS must be at least 1 and F_PEAK_HZ positive')

def smooth(v, n):
    """v averaged with its four neighbours n times, the edges held: the earth without its sharp edges."""
    for k in range(n):
        p = np.pad(v, 1, mode='edge')
        v = (p[1:-1, 1:-1] + p[:-2, 1:-1] + p[2:, 1:-1] + p[1:-1, :-2] + p[1:-1, 2:]) / 5.0
    return v

def field_shot(window, f_peak, frames=None, every_ms=0.0):
    """One shot at the centre of a window of the field earth, on square cells by the book's rule, and the same shot
    through the same earth smoothed: the window (m/s), the cell (m), the step (s), and the two records. Given a list
    frames, the wavefield is kept every every_ms."""
    f = min(f_peak, FIELD_F_MAX_HZ)
    h = alford_h(field_earth()['vp'].min(), f)
    e = field_earth(nx=int(3600.0 / h), nz=int(1600.0 / h), dx=h, dz=h, seal=ROCKS['shale'], upper=ROCKS['gas sand'],
                    lower=ROCKS['brine sand'])
    a, b = WINDOWS[window]
    v = e['vp'][:int(FIELD_DEPTH_M / h), int(a * e['x'].size):int(b * e['x'].size)]
    vp = np.pad(v, N_ABS, mode='edge')                     # the sponge's cells, outside the window
    dt = 0.95 * COURANT * h / vp.max()
    nt = int(FIELD_T_MS * 1e-3 / dt)
    src, n0 = source_derivative(f, dt, nt)
    iz, ix, rx = N_ABS + 2, N_ABS + v.shape[1] // 2, np.arange(N_ABS, N_ABS + v.shape[1])
    rec = run(vp, h, dt, nt, iz, ix, src, iz, rx, frames, max(1, int(every_ms * 1e-3 / dt)))
    return dict(v=v, x0=a * e['x'][-1], h=h, f=f, dt=dt, nt=nt, rec=rec, bg=run(smooth(vp, SMOOTH_STEPS), h, dt, nt, iz, ix, src, iz, rx))

def plot_shot(s, snap, snap_ms, title_bg):
    """The wavefield at snap_ms over the window's layering, the record, and what the structure adds
    to it: the record minus the smooth earth's, muted above the direct wave. The records carry a gain of time squared."""
    fig, ax = plt.subplots(1, 3, figsize=(13.5, 4.5), gridspec_kw={'width_ratios': [1.25, 1, 1]})
    v, h = s['v'], s['h']
    xk = (s['x0'] + np.arange(v.shape[1]) * h) / 1e3
    zm = np.arange(v.shape[0]) * h
    u = snap[N_ABS:N_ABS + v.shape[0], N_ABS:N_ABS + v.shape[1]]
    c = np.percentile(np.abs(u), 99.0) + 1e-30
    ax[0].imshow(u, cmap='seismic', vmin=-c, vmax=c, aspect='auto', extent=[xk[0], xk[-1], zm[-1], 0])
    ax[0].contour(xk, zm, v, levels=12, colors='0.35', linewidths=0.4)
    style(ax[0], 'The wavefield at %.0f ms, over the layering' % snap_ms, 'distance (km)', 'depth (m)', grid=False)
    t_ms = np.arange(s['nt']) * s['dt'] * 1000.0
    gain = (t_ms[:, None] / 1000.0) ** 2
    mute = t_ms[:, None] > 1e6 * np.abs(xk - xk[xk.size // 2])[None, :] / MUTE_MPS + MUTE_MS     # below the direct wave
    for a, r, name in ((ax[1], s['rec'], 'The shot record'), (ax[2], (s['rec'] - s['bg']) * mute, 'Minus the record of ' + title_bg)):
        r = r * gain
        c = np.abs(r).max() * CLIP
        a.imshow(r, cmap='gray_r', vmin=-c, vmax=c, aspect='auto', extent=[xk[0], xk[-1], t_ms[-1], 0])
        style(a, name, 'receiver position (km)', 'time (ms), gain t squared', grid=False)
    fig.tight_layout()
    plt.show()

def main():
    validate(GRID_N, F_PEAK_HZ, N_SHOTS)
    vp_shale = ROCKS['shale'][0]
    # 1. the size of the full job, counted the way Figure 0.2 counts (multiply-adds, no sponge)
    h, dt_ms, steps, cells, ops = job_size(GRID_N, F_PEAK_HZ, VP_MIN, VP_MAX, N_SHOTS)
    report('cell_m', h)
    report('dt_ms', dt_ms)
    report('steps', steps, '%d')
    report('cells', cells, '%d')
    report('multiply_adds', ops, '%d')
    report('with_sponge_multiply_adds', (GRID_N + 2 * N_ABS) ** 2 * steps * NPTS * N_SHOTS, '%d')
    # 2. the check: a plane wave in a layered column must arrive when the travel time says
    t_sim, t_pred = plane_wave_check(vp_shale, ROCKS['brine sand'][0], F_PEAK_HZ, h, dt_ms)
    report('plane_wave_reflection_ms', t_sim, '%.1f')
    report('plane_wave_predicted_ms', t_pred, '%.1f')
    # self-check: the stencil puts the plane wave's reflection where the travel time does
    assert abs(t_sim - t_pred) / t_pred < 0.006
    # 3. the field: one shot through a window of the field earth (the job above is counted, this is run)
    frames = []                                            # the wavefield at the start, at SNAP_MS, at twice it...
    s = field_shot(WINDOW, F_PEAK_HZ, frames, SNAP_MS)
    fieldnote('window_km', s['v'].shape[1] * s['h'] / 1e3, '%.2f')
    fieldnote('peak_hz', s['f'], '%.1f')
    fieldnote('cell_m', s['h'], '%.2f')
    fieldnote('dt_ms', s['dt'] * 1e3, '%.3f')
    fieldnote('steps', s['nt'], '%d')
    fieldnote('cells_with_sponge', (s['v'].shape[0] + 2 * N_ABS) * (s['v'].shape[1] + 2 * N_ABS), '%d')
    fieldnote('structure_over_record', np.abs(s['rec'] - s['bg']).max() / np.abs(s['rec']).max())
    plot_shot(s, frames[1], 1000.0 * s['dt'] * max(1, int(SNAP_MS * 1e-3 / s['dt'])), TITLE_BG)

if __name__ == '__main__':
    main()

## Where to go next

- The course: https://ogbonlab.com/textbooks/synthetic-seismic-modeling
- More notebooks: https://github.com/mosesfalowo/ogbonlab-notebooks
- Scale up: Devito, SPECFEM3D, Madagascar (course Part 12.3)